# Interval scorers
Skchange provides modular and flexible algorithms for change detection that also run fast.
*Interval scorers* are the components that make this possible.
They are user-specified building blocks of all change detectors in Skchange,
and they represent the distributional feature(s) of the data that the detector is looking for changes in,
like the mean, variance, covariance matrix, regression parameters, the full distribution, a Poisson rate, or something else.

An interval scorer is responsible for evaluating a *score* over *interval specifications* (*interval specs* for short).
An interval spec consists of the start and end indices of an interval, and possibly split points inside it.
Different detectors require different types of scores. Below is a summary of the four types of interval scorers in Skchange, and examples of which detectors use them (more on each type below):

| Score type | Interval spec| What it scores | Example detectors |
|---|---|---|---|
| Cost | start, end | The cost/loss of a model fit to a single interval `[start, end)`. | [PELT](../../api_reference/auto_generated/skchange.detectors.PELT.rst), [CROPS](../../api_reference/auto_generated/skchange.detectors.CROPS.rst) |
| Change score | start, split, end | The degree of change between two adjacent intervals `[start, split)` and `[split, end)`. A two-sample statistical test, also known as an "at-most-one-change" test statistic. | [SeededBinarySegmentation](../../api_reference/auto_generated/skchange.detectors.SeededBinarySegmentation.rst), [MovingWindow](../../api_reference/auto_generated/skchange.detectors.MovingWindow.rst) |
| Saving | start, end | The degree to which an interval `[start, end)` deviates from a fixed baseline model. A one-sample statistical test. | [CAPA](../../api_reference/auto_generated/skchange.detectors.CAPA.rst) |
| Transient score | start, split1, split2, end | The degree to which an interval `[split1, split2)` is different compared to its local context `[start, split1)` and `[split2, end)`. A two-sample statistical test, but with a different splitting strategy than change scores. | [CircularBinarySegmentation](../../api_reference/auto_generated/skchange.detectors.CircularBinarySegmentation.rst) |

The computational bottleneck of most change detection algorithms is to evaluate the score over a large number of interval specs, often with overlapping regions. The design of interval scorers provides two mechanisms for speeding this up:

1. **Precomputation**: Interval scorers can precompute quantities that are reused across many interval specs, such as cumulative sums or sufficient statistics.
2. **Vectorisation**: Interval scorers can evaluate many interval specs in a single call, often using `numba` for acceleration.

## API
As a user, you only need to construct an interval scorer, while its methods are called internally by the detector.
To make full use of the library, however, it is useful to understand how they work and how they can be used to build more complex detectors.

All interval scorers share the same three-step usage:

* `fit(X)`: Fits the scorer to training data. This only validates the data and parameters in many cases, but not all. This is called within the detector's `fit` method.
* `precompute(X)`: Takes (possibly) new data and returns a cache of reusable quantities.
* `evaluate(cache, interval_specs)`: Scores a batch of interval specs in one call.

Like detectors, interval scorers inherit from Scikit-learn's `BaseEstimator`.
This enables seamless use of `get_params`, `set_params`, and `clone` for hyperparameter inspection, updating, and copying.

## Cost

A *cost* measures the cost/loss/error of a model fit to a data interval `X[start:end]`.
For example, [L2Cost](../../api_reference/auto_generated/skchange.interval_scorers.L2Cost.rst) returns the sum of squared deviations from the sample mean within each interval. This sum is small when the interval is well-modelled by a single mean and large when it straddles a change.

Consider an example univariate series of length 30 with a mean change at index 20:

In [ ]:
from skchange.datasets import generate_piecewise_normal_data
from skchange.interval_scorers import L2Cost

X = generate_piecewise_normal_data(means=[0, 5], lengths=[20, 10], seed=1)

cost = L2Cost().fit(X)
cache = cost.precompute(X)
cost.evaluate(cache, [[0, 5], [15, 25], [25, 30]])

The L2 cost is much larger for the interval `[15, 25)` than for the other two, because that interval straddles the change point at index 20 and a single-mean model fits poorly there.

## Change score

A *change score* takes in a triplet `(start, split, end)` and measures the degree of change between the two adjacent intervals `X[start:split]` and `X[split:end]`. Change scores can be statistical tests, time-series distances, or any other measure of difference. A classical example is the [CUSUM](../../api_reference/auto_generated/skchange.interval_scorers.CUSUM.rst) score for a change in mean.

In [ ]:
from skchange.interval_scorers import CUSUM

score = CUSUM().fit(X)
cache = score.precompute(X)
score.evaluate(cache, [[0, 3, 6], [15, 20, 25], [20, 25, 30]])

Again, the change score is largest for the interval that contains the change point at index 20.

## Saving
A *saving* takes a `(start, end)` spec and measures the difference in cost between a locally estimated and a fixed reference model parameter over `X[start:end]`. The reference parameter represents the "normal" data behaviour and is either user-specified or estimated robustly from the training data in the saving's `fit` method. Savings are used primarily for segment anomaly detection.

In [ ]:
from skchange.interval_scorers import L2Saving

saving = L2Saving(baseline_mean=None)  # Estimate the baseline mean robustly in fit.
saving.fit(X)
print(f"Baseline mean: {saving.baseline_mean_[0]:.3f}")

cache = saving.precompute(X)
saving.evaluate(cache, [[0, 5], [15, 25], [20, 30]])

The saving is largest for the last interval, whose mean differs most from the baseline.

## Transient score

A *transient score* takes a quadruplet `(start, split1, split2, end)` and measures how different the inner interval `X[split1:split2]` is compared to its local context to the left and right, `X[start:split1]` and `X[split2:end]`.
These scores serve a similar purpose to savings and are primarily used for segment anomaly detection, but they don't require a fixed baseline model at the cost of often being heavier to compute.

As an example, [L2TransientScore](../../api_reference/auto_generated/skchange.interval_scorers.L2TransientScore.rst) measures the reduction in squared error obtained by letting the inner interval have one mean and the surrounding intervals a separate mean, relative to the whole outer interval `X[start:end]` sharing a single mean.

In [ ]:
from skchange.interval_scorers import L2TransientScore

transient_score = L2TransientScore().fit(X)
cache = transient_score.precompute(X)
transient_score.evaluate(cache, [[0, 5, 10, 15], [15, 20, 25, 30], [0, 20, 30, 30]])

The transient score is largest for the last interval, whose inner region `[20, 30)` sits entirely in the post-change segment while the surrounding context `[0, 20)` sits entirely in the pre-change segment. The first interval is homogeneous and scores near zero. The second sits across the change point, but its surrounding context mixes pre- and post-change data, so the contrast is smaller.